In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import re
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from bs4 import BeautifulSoup
import re

In [2]:
data = pd.read_csv('/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/sample-data.csv')

In [3]:
data.head()

,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."


In [4]:
sample = data.sample(n=5, random_state=42)

In [5]:
pd.set_option('display.max_colwidth', None)
display(sample)

,id,description
361,362,"S/s sol patrol shirt - In theory, chasing Indonesian surf and bonefish for weeks on end is like going to heaven; in practice, it's really damn hot. To make sure you're covered, we've travel-tested the Short-Sleeved Sol Patrol Shirt for function-forward UV protection in sweltering spots the world over. The 2.4-oz 100% polyester ripstop fabric and side air vents are receptive to any breeze providing comfort and hardcore 30-UPF sun protection in a wide array of circumstances. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Lightweight, quick-drying polyester""</li> <li>Provides 30-UPF sun protection</li> <li>Double-layer collar converts to stand-up collar for extra sun protection</li> <li>Collar stays put and crisp thanks to hidden snaps at collar points</li> <li>Cooling mesh center back vent and side-arm vents facilitate flow-through ventilation</li> <li>Mesh-lined button-through chest pockets; inset zippered pocket at right</li></ul><br><br><b>Fabric: </b>2.4-oz 100% polyester ripstop with 30-UPF sun protection. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>(216 g 7.5 oz)<br><br>Made in Vietnam."
73,74,"Logo hat - Your unflappable road-tripping companion - this 100% organic cotton hat disguises bed head, cuts glare when you're checking surf, and pulls chinward for an afternoon siesta. The go-anywhere design has a precurved brim, a rear elastic strap with nylon webbing for adjustability and a moisture-wicking headband for comfort. The low profile ensures a reliable bond during rogue gusts.<br><br><b>Details:</b><ul> <li>""Comfortable, 100% organic cotton fabric holds its shape""</li> <li>Shapeable brim shades face and reduces glare</li> <li>Rear webbing strap with adjustable buckle</li> <li>Moisture-wicking headband</li></ul><br><br><b>Fabric: </b>5.2-oz 100% organic cotton canvas<br><br><b>Weight: </b>(77 g 2.7 oz)<br><br>Made in China."
374,375,"Pique vitaliti polo shirt - The perfect summer fabric to keep your back from sticking to the seat. The pique polo's forward-thinking blend of 6.5-oz 70% organic cotton and 30% all-recycled polyester also wicks moisture during long walks beneath the Flatirons, highball V4s at Flagstaff and hardcore sodbusting sessions in your own back yard.<br><br><b>Details:</b><ul> <li>""Lightweight, quick-to dry organic cotton/recycled polyester fabric in a piqu? knit""</li> <li>Hidden snap-front placket</li> <li>Side vents</li></ul><br><br><b>Fabric: </b>6.5-oz 70% organic cotton/30% all-recycled polyester piqu??. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>(262 g 9.1 oz)<br><br>Made in Thailand."
155,156,"Active mesh bra - Despite its inability to act as a belay slave, our Active Mesh Bra makes a perfect climbing partner. The body-conforming fabric provides fluid support and wicks moisture, while the seamless construction eliminates chafing. Comfortable nonslip straps won't dig into skin; they contour to a flattering V in back for unobstructed movement. The full-coverage design can go solo, or partner with layers. Made from 95% nylon mesh/5% spandex.<br><br><b>Details:</b><ul> <li>""Stretchy, body-conforming nylon/spandex blend is ultradurable, supportive and wicks moisture to keep skin dry""</li> <li>Seamless construction</li> <li>""Comfortable, nonslip straps meet V-style in back""</li></ul><br><br><b>Fabric: </b>8.1-oz 95% nylon mesh/5% spandex with moisture-wicking performance<br><br><b>Weight: </b>(72 g 2.5 oz)<br><br>Made in Israel."
104,105,"R1 hoody - Spotted round the globe from high alpine ridges to backcountry bowls, our R1 Hoody earns raves from our most active customers and ambassadors alike. Warm, breathable and mobile, it has the minimalist detailing that avid climbers and backcountry skiers need - from the stretchy R1 fabric (93% all-recycled polyester/7% spandex) to the snug-fitting balaclava-style hood. With low-grid Capilene 4 stretch polyester unde

In [6]:
sample_clean = sample.copy()

# Créer un objet BeautifulSoup pour chaque description
html = sample_clean["description"].fillna("").apply(
    lambda texte: BeautifulSoup(texte, "html.parser")
)

In [7]:
def extraire_description(soup):
    """Récupérer le texte avant la première balise <br> ou <b>."""
    morceaux = []

    for element in soup.contents:
        if element.name in ["br", "b"]:
            break

        morceaux.append(str(element))

    texte_html = " ".join(morceaux)
    return BeautifulSoup(texte_html, "html.parser").get_text(" ", strip=True)


sample_clean["description_texte"] = html.apply(extraire_description)

In [8]:
def extraire_details(soup):
    """Récupérer les éléments de la liste Details."""
    titre = soup.find("b", string=re.compile(r"^\s*Details\s*:\s*$", re.I))

    if titre is None:
        return []

    liste = titre.find_next_sibling()

    if liste is None or liste.name != "ul":
        return []

    details = []

    for element in liste.find_all("li"):
        details.append(element.get_text(" ", strip=True))

    return details


sample_clean["details"] = html.apply(extraire_details)

In [9]:
def extraire_valeur(soup, nom):
    """Récupérer le texte qui suit un titre en gras."""
    motif = rf"^\s*{re.escape(nom)}\s*:\s*$"
    titre = soup.find("b", string=re.compile(motif, re.I))

    if titre is None:
        return None

    morceaux = []

    for element in titre.next_siblings:
        if element.name in ["br", "b"]:
            break

        morceaux.append(str(element))

    texte_html = " ".join(morceaux)
    return BeautifulSoup(texte_html, "html.parser").get_text(" ", strip=True)


sample_clean["fabric"] = html.apply(lambda soup: extraire_valeur(soup, "Fabric"))
sample_clean["weight"] = html.apply(lambda soup: extraire_valeur(soup, "Weight"))

In [10]:
def extraire_pays(soup):
    """Récupérer le pays après « Made in »."""
    for texte in soup.stripped_strings:
        if texte.lower().startswith("made in "):
            return texte[8:].strip().rstrip(".")

    return None


sample_clean["pays"] = html.apply(extraire_pays)

In [11]:
display(sample_clean[[
    "description_texte",
    "details",
    "fabric",
    "weight",
    "pays"
]])

,description_texte,details,fabric,weight,pays
361,"S/s sol patrol shirt - In theory, chasing Indonesian surf and bonefish for weeks on end is like going to heaven; in practice, it's really damn hot. To make sure you're covered, we've travel-tested the Short-Sleeved Sol Patrol Shirt for function-forward UV protection in sweltering spots the world over. The 2.4-oz 100% polyester ripstop fabric and side air vents are receptive to any breeze providing comfort and hardcore 30-UPF sun protection in a wide array of circumstances. Recyclable through the Common Threads Recycling Program.","[""Lightweight, quick-drying polyester"", Provides 30-UPF sun protection, Double-layer collar converts to stand-up collar for extra sun protection, Collar stays put and crisp thanks to hidden snaps at collar points, Cooling mesh center back vent and side-arm vents facilitate flow-through ventilation, Mesh-lined button-through chest pockets; inset zippered pocket at right]",2.4-oz 100% polyester ripstop with 30-UPF sun protection. Recyclable through the Common Threads Recycling Program,(216 g 7.5 oz),Vietnam
73,"Logo hat - Your unflappable road-tripping companion - this 100% organic cotton hat disguises bed head, cuts glare when you're checking surf, and pulls chinward for an afternoon siesta. The go-anywhere design has a precurved brim, a rear elastic strap with nylon webbing for adjustability and a moisture-wicking headband for comfort. The low profile ensures a reliable bond during rogue gusts.","[""Comfortable, 100% organic cotton fabric holds its shape"", Shapeable brim shades face and reduces glare, Rear webbing strap with adjustable buckle, Moisture-wicking headband]",5.2-oz 100% organic cotton canvas,(77 g 2.7 oz),China
374,"Pique vitaliti polo shirt - The perfect summer fabric to keep your back from sticking to the seat. The pique polo's forward-thinking blend of 6.5-oz 70% organic cotton and 30% all-recycled polyester also wicks moisture during long walks beneath the Flatirons, highball V4s at Flagstaff and hardcore sodbusting sessions in your own back yard.","[""Lightweight, quick-to dry organic cotton/recycled polyester fabric in a piqu? knit"", Hidden snap-front placket, Side vents]",6.5-oz 70% organic cotton/30% all-recycled polyester piqu??. Recyclable through the Common Threads Recycling Program,(262 g 9.1 oz),Thailand
155,"Active mesh bra - Despite its inability to act as a belay slave, our Active Mesh Bra makes a perfect climbing partner. The body-conforming fabric provides fluid support and wicks moisture, while the seamless construction eliminates chafing. Comfortable nonslip straps won't dig into skin; they contour to a flattering V in back for unobstructed movement. The full-coverage design can go solo, or partner with layers. Made from 95% nylon mesh/5% spandex.","[""Stretchy, body-conforming nylon/spandex blend is ultradurable, supportive and wicks moisture to keep skin dry"", Seamless construction, ""Comfortable, nonslip straps meet V-style in back""]",8.1-oz 95% nylon mesh/5% spandex with moisture-wicking performance,(72 g 2.5 oz),Israel
104,"R1 hoody - Spotted round the globe from high alpine ridges to backcountry bowls, our R1 Hoody earns raves from our most active customers and ambassadors alike. Warm, breathable and mobile, it has the minimalist detailing that avid climbers and backcountry skiers need - from the stretchy R1 fabric (93% all-recycled polyester/7% spandex) to the snug-fitting balaclava-style hood. With low-grid Capilene 4 stretch polyester under the arms, at the wrist cuffs and around the bottom hem for minimal bulk under a harness. Recyclable through the Common Threads Recycling Program.","[""Versatile R1 fleece provides excellent stretch, warmth, wicking and breathability in a variety of temperatures"", ""High/low interior grid-structure enhances compressibility, airflow and dry time"", Jersey microfiber face allows nonbinding layering, ""Capilene 4 stretch panels under arms, at cuffs and hem for reduced b

In [12]:
sample_clean["details"] = sample_clean["details"].apply(
    lambda liste: " ".join(liste)
)

In [13]:
def nettoyer_texte(texte):
    texte = texte.lower()
    texte = re.sub(r"[^\w\s%/.,-]", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()

In [14]:
for col in ["description_texte", "details", "fabric", "weight"]:
    sample_clean[col] = sample_clean[col].apply(
        lambda texte: nettoyer_texte(texte) if pd.notna(texte) else texte
    )

In [16]:
display(sample_clean)

,id,description,description_texte,details,fabric,weight,pays
361,362,"S/s sol patrol shirt - In theory, chasing Indonesian surf and bonefish for weeks on end is like going to heaven; in practice, it's really damn hot. To make sure you're covered, we've travel-tested the Short-Sleeved Sol Patrol Shirt for function-forward UV protection in sweltering spots the world over. The 2.4-oz 100% polyester ripstop fabric and side air vents are receptive to any breeze providing comfort and hardcore 30-UPF sun protection in a wide array of circumstances. Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>""Lightweight, quick-drying polyester""</li> <li>Provides 30-UPF sun protection</li> <li>Double-layer collar converts to stand-up collar for extra sun protection</li> <li>Collar stays put and crisp thanks to hidden snaps at collar points</li> <li>Cooling mesh center back vent and side-arm vents facilitate flow-through ventilation</li> <li>Mesh-lined button-through chest pockets; inset zippered pocket at right</li></ul><br><br><b>Fabric: </b>2.4-oz 100% polyester ripstop with 30-UPF sun protection. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>(216 g 7.5 oz)<br><br>Made in Vietnam.","s/s sol patrol shirt - in theory, chasing indonesian surf and bonefish for weeks on end is like going to heaven in practice, it s really damn hot. to make sure you re covered, we ve travel-tested the short-sleeved sol patrol shirt for function-forward uv protection in sweltering spots the world over. the 2.4-oz 100% polyester ripstop fabric and side air vents are receptive to any breeze providing comfort and hardcore 30-upf sun protection in a wide array of circumstances. recyclable through the common threads recycling program.","lightweight, quick-drying polyester provides 30-upf sun protection double-layer collar converts to stand-up collar for extra sun protection collar stays put and crisp thanks to hidden snaps at collar points cooling mesh center back vent and side-arm vents facilitate flow-through ventilation mesh-lined button-through chest pockets inset zippered pocket at right",2.4-oz 100% polyester ripstop with 30-upf sun protection. recyclable through the common threads recycling program,216 g 7.5 oz,Vietnam
73,74,"Logo hat - Your unflappable road-tripping companion - this 100% organic cotton hat disguises bed head, cuts glare when you're checking surf, and pulls chinward for an afternoon siesta. The go-anywhere design has a precurved brim, a rear elastic strap with nylon webbing for adjustability and a moisture-wicking headband for comfort. The low profile ensures a reliable bond during rogue gusts.<br><br><b>Details:</b><ul> <li>""Comfortable, 100% organic cotton fabric holds its shape""</li> <li>Shapeable brim shades face and reduces glare</li> <li>Rear webbing strap with adjustable buckle</li> <li>Moisture-wicking headband</li></ul><br><br><b>Fabric: </b>5.2-oz 100% organic cotton canvas<br><br><b>Weight: </b>(77 g 2.7 oz)<br><br>Made in China.","logo hat - your unflappable road-tripping companion - this 100% organic cotton hat disguises bed head, cuts glare when you re checking surf, and pulls chinward for an afternoon siesta. the go-anywhere design has a precurved brim, a rear elastic strap with nylon webbing for adjustability and a moisture-wicking headband for comfort. the low profile ensures a reliable bond during rogue gusts.","comfortable, 100% organic cotton fabric holds its shape shapeable brim shades face and reduces glare rear webbing strap with adjustable buckle moisture-wicking headband",5.2-oz 100% organic cotton canvas,77 g 2.7 oz,China
374,375,"Pique vitaliti polo shirt - The perfect summer fabric to keep your back from sticking to the seat. The pique polo's forward-thinking blend of 6.5-oz 70% organic cotton and 30% all-recycled polyester also wicks moisture during long walks beneath the Flatirons, highball V4s at Flagstaff and hardcore sodbusting sessions in yo

In [ ]:
sample_clean.drop(columns=["description"], inplace=True)

In [19]:
sample_clean

,id,description_texte,details,fabric,weight,pays
361,362,"s/s sol patrol shirt - in theory, chasing indonesian surf and bonefish for weeks on end is like going to heaven in practice, it s really damn hot. to make sure you re covered, we ve travel-tested the short-sleeved sol patrol shirt for function-forward uv protection in sweltering spots the world over. the 2.4-oz 100% polyester ripstop fabric and side air vents are receptive to any breeze providing comfort and hardcore 30-upf sun protection in a wide array of circumstances. recyclable through the common threads recycling program.","lightweight, quick-drying polyester provides 30-upf sun protection double-layer collar converts to stand-up collar for extra sun protection collar stays put and crisp thanks to hidden snaps at collar points cooling mesh center back vent and side-arm vents facilitate flow-through ventilation mesh-lined button-through chest pockets inset zippered pocket at right",2.4-oz 100% polyester ripstop with 30-upf sun protection. recyclable through the common threads recycling program,216 g 7.5 oz,Vietnam
73,74,"logo hat - your unflappable road-tripping companion - this 100% organic cotton hat disguises bed head, cuts glare when you re checking surf, and pulls chinward for an afternoon siesta. the go-anywhere design has a precurved brim, a rear elastic strap with nylon webbing for adjustability and a moisture-wicking headband for comfort. the low profile ensures a reliable bond during rogue gusts.","comfortable, 100% organic cotton fabric holds its shape shapeable brim shades face and reduces glare rear webbing strap with adjustable buckle moisture-wicking headband",5.2-oz 100% organic cotton canvas,77 g 2.7 oz,China
374,375,"pique vitaliti polo shirt - the perfect summer fabric to keep your back from sticking to the seat. the pique polo s forward-thinking blend of 6.5-oz 70% organic cotton and 30% all-recycled polyester also wicks moisture during long walks beneath the flatirons, highball v4s at flagstaff and hardcore sodbusting sessions in your own back yard.","lightweight, quick-to dry organic cotton/recycled polyester fabric in a piqu knit hidden snap-front placket side vents",6.5-oz 70% organic cotton/30% all-recycled polyester piqu . recyclable through the common threads recycling program,262 g 9.1 oz,Thailand
155,156,"active mesh bra - despite its inability to act as a belay slave, our active mesh bra makes a perfect climbing partner. the body-conforming fabric provides fluid support and wicks moisture, while the seamless construction eliminates chafing. comfortable nonslip straps won t dig into skin they contour to a flattering v in back for unobstructed movement. the full-coverage design can go solo, or partner with layers. made from 95% nylon mesh/5% spandex.","stretchy, body-conforming nylon/spandex blend is ultradurable, supportive and wicks moisture to keep skin dry seamless construction comfortable, nonslip straps meet v-style in back",8.1-oz 95% nylon mesh/5% spandex with moisture-wicking performance,72 g 2.5 oz,Israel
104,105,"r1 hoody - spotted round the globe from high alpine ridges to backcountry bowls, our r1 hoody earns raves from our most active customers and ambassadors alike. warm, breathable and mobile, it has the minimalist detailing that avid climbers and backcountry skiers need - from the stretchy r1 fabric 93% all-recycled polyester/7% spandex to the snug-fitting balaclava-style hood. with low-grid capilene 4 stretch polyester under the arms, at the wrist cuffs and around the bottom hem for minimal bulk under a harness. recyclable through the common threads recycling program.","versatile r1 fleece provides excellent stretch, warmth, wicking and breathability in a variety of temperatures high/low interior grid-structure enhances compressibility, airflow and dry time jersey microfiber face allows nonbinding layering capilene 4 stretch panels under arms, at cuffs and hem for reduced bulk and increased breathability snug-fitting ba

In [17]:
sample_clean.to_csv('/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/Nico/sample_clean.csv', index=False)